# Experiment 2 — learning curves (PD)

How AUC scales with the amount of available data (no HPO; LogReg, TabICLv2, TabPFN-v3, and CatBoost). The swept `row_limit` caps the **dataset before the CV split** — train and test shrink together — so the x-axis is the **dataset size**, not the training-set size. Pooled curves in several views, then per-dataset raw points, a data-efficiency table, and a summary of the **AUC trajectory** across the whole sweep. Figures → `figures/experiment2/pd/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.methods.method_config import HPO_METHODS          # tuned methods -> HPO time x n_trials
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, rank_heatmap, hpo_improvement_bars,
    compute_time_bars, runtime_performance_scatter,
    learning_curve, learning_curve_moving_average_with_dots,
    imbalance_curve, imbalance_curve_moving_average_with_dots, per_dataset_sweep_curves,
    sweep_evolution_summary, pd_summary_text, lgd_summary_text,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment2/pd')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df = load_summary(SUMMARY_DIR, experiment='experiment2', task='pd', aggregated=False)
DISPLAY_METHODS = ['LogReg', 'tabicl_v2', 'tabpfn_v3', 'catboost']
missing = sorted(set(DISPLAY_METHODS) - set(df['method']))
if missing:
    print('Missing display methods:', ', '.join(missing))
df = df[df['method'].isin(DISPLAY_METHODS)].copy()
print(f'{df["method"].nunique()} methods, {df["dataset"].nunique()} datasets, '
      f'{df["sweep_value"].nunique()} sweep points')

## 1. AUC vs dataset size ? pooled over datasets

Views of the same curve (one line per method, mean over datasets):
1. **learning curve** with raw sweep points,
2. the same learning curve with a **lower-right inset** zooming the shaded `rows <= 1000` region,
3. **moving average** (trend),
4. **moving average with transparent raw points**.


In [ ]:
learning_curve(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)


In [ ]:
learning_curve(df, 'AUC', task_name='PD', zoom=True, out_dir=FIGURES_DIR)


In [ ]:
for smooth_window in ('less', 'standard', 'more'):
    learning_curve(df, 'AUC', task_name='PD', smooth=True, smooth_window=smooth_window, out_dir=FIGURES_DIR)


In [ ]:
for smooth_window in ('less', 'standard', 'more'):
    learning_curve_moving_average_with_dots(df, 'AUC', task_name='PD', smooth_window=smooth_window, out_dir=FIGURES_DIR)


## 2. AUC per dataset (raw points, all methods)

One plot per dataset so per-dataset behaviour is visible, not only the pooled mean.

In [ ]:
per_dataset_sweep_curves(df, 'AUC', sweep_axis='row_limit', xlabel='Dataset size (rows)', task_name='PD', out_dir=FIGURES_DIR)

## 3. Data efficiency — dataset rows to reach 95% of each method's own plateau

In [ ]:
import pandas as _pd
g = df.groupby(['method','sweep_value'])['metric.AUC'].mean().reset_index()
rows = {}
for m, gg in g.groupby('method'):
    gg = gg.sort_values('sweep_value'); target = gg['metric.AUC'].max() * 0.95
    hit = gg[gg['metric.AUC'] >= target]
    rows[m] = int(hit['sweep_value'].iloc[0]) if len(hit) else None
display(_pd.Series(rows, name='dataset rows to reach 95% of own max AUC').sort_values())

## 4. Summary — AUC evolution over dataset size

Mean AUC per method across all datasets at a spread of dataset sizes (not just the largest), so the whole trajectory is visible.

In [ ]:
sweep_evolution_summary(df, 'AUC', sweep_axis='row_limit',
                        task_name='Experiment 2 — PD', log_spacing=True, n_points=18)